# Hydrocephalus: logistic regression with the repository’s original cross-validation

This runs one model with the repository’s default 10-fold × 10-repeat outer validation and 10-fold inner tuning. It uses the exact commit captured in the successful smoke test. The full study includes other classifiers and outputs; this notebook does not reproduce the complete paper. Run in Google Colab with CPU. No laptop installation. The final training cell may take substantial time.


In [ ]:
import subprocess, sys
from pathlib import Path
repo = Path('/content/shuntclf')
env_python = Path('/content/shunt-env/bin/python')
commit = '1c0ef7020fd2437dc54bc685d8c178b452988ec9'
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/RISCSoftware/shuntclf.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', commit], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', commit], check=True)
actual = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == commit
print('Repository commit:', actual)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'python', 'install', '3.11'], check=True)
if not env_python.exists():
    subprocess.run(['uv', 'venv', '--python', '3.11', '/content/shunt-env'], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', str(env_python), '-r', str(repo / 'requirements.txt')], check=True)
subprocess.run([str(env_python), '--version'], check=True)


## Train and evaluate logistic regression

The evaluation calls the repository’s `train_one` function with the same split counts, seeds, data processing, model, and scoring used by its `main` function. Only the parallel worker count is reduced to 2 for a Colab CPU runtime.


In [ ]:
from textwrap import dedent
import subprocess
script = dedent("import hashlib, json, subprocess, sys\nfrom datetime import datetime, timezone\nfrom importlib.metadata import version\nfrom pathlib import Path\nfrom sklearn.model_selection import RepeatedStratifiedKFold\nfrom shuntclf.classifier_pipelines import ClfCode\nfrom shuntclf.dataset import AneurysmLocationMapper1, get_dataset\nfrom shuntclf.main import train_one\nrepo = Path('/content/shuntclf')\ncommit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()\nassert commit == '1c0ef7020fd2437dc54bc685d8c178b452988ec9'\ndata_file = repo / 'data' / 'dataset.csv'\ndata_sha256 = hashlib.sha256(data_file.read_bytes()).hexdigest()\ndf, features, target = get_dataset(aneurysm_location_mapper=AneurysmLocationMapper1, only_with_aneurysm=False)\nassert len(df) == 510\nstart = datetime.now(timezone.utc).isoformat()\nprint('Starting 100 outer folds; each includes 10-fold hyperparameter tuning.')\nprint('This may take a while in Colab. Leave the session connected.')\nouter = RepeatedStratifiedKFold(n_splits=10, n_repeats=10, random_state=0)\ninner = RepeatedStratifiedKFold(n_splits=10, n_repeats=1, random_state=1)\nscores, stats = train_one(clf_code=ClfCode.logistic_regression, split=outer, inner_split=inner, aneurysm_location_mapper=AneurysmLocationMapper1, only_with_aneurysm=False, random_state=0, return_estimators=False, return_curves=False, return_feature_importance=False, n_jobs=2)\nfinish = datetime.now(timezone.utc).isoformat()\nresult = {\n    'status': 'single_model_original_cv_completed',\n    'scope': 'Repository train_one logic for logistic regression only; not all classifiers or all paper outputs',\n    'repository': 'https://github.com/RISCSoftware/shuntclf',\n    'repository_commit': commit,\n    'dataset_sha256': data_sha256,\n    'python': sys.version.split()[0],\n    'versions': {x: version(x) for x in ['numpy','pandas','scikit-learn','imbalanced-learn','xgboost']},\n    'patients': len(df), 'processed_features': len(features),\n    'target_counts': {str(k): int(v) for k,v in df[target].value_counts().items()},\n    'model': 'logistic_regression',\n    'outer_folds': 10, 'outer_repeats': 10,\n    'inner_folds': 10, 'inner_repeats': 1, 'random_seed': 0,\n    'started_utc': start, 'finished_utc': finish,\n    'test_metrics': {k: float(v) for k,v in stats.items() if k.startswith('test_')},\n    'test_scores_by_fold': {k: [float(v) for v in vals] for k,vals in scores.items() if k.startswith('test_')},\n}\npath = Path('/content/shuntclf_logistic_original_cv_results.json')\npath.write_text(json.dumps(result, indent=2) + '\\n')\nprint('Finished. Key test metrics:')\nfor key in ['test_roc_auc_mean','test_pr_auc_mean','test_balanced_acc_mean','test_f1_mean']:\n    print(key, result['test_metrics'][key])\nprint('Saved:', path)\n")
subprocess.run([str(env_python), '-c', script], cwd=str(repo), check=True)


## Download the evidence

After successful completion, download the JSON. It contains the commit, dataset hash, package versions, exact CV settings, summary scores and scores for all 100 outer folds.


In [ ]:
from google.colab import files
files.download('/content/shuntclf_logistic_original_cv_results.json')
